In [12]:
import json
from typing import List
import numpy as np
import librosa


def load_audio(path: str, sr: int = 16000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y.astype(np.float32), sr

def load_subset(subset_path: str) -> List[dict]:
    with open(subset_path, "r", encoding="utf-8") as f:
        subset = json.load(f)
    if not isinstance(subset, list):
        raise ValueError("Subset file must contain a JSON list.")
    return subset

subset_path = '../../dcase_subset.json'

subset = load_subset(subset_path)
import random
random.shuffle(subset)

In [13]:
import sys
sys.path.append('../../')
from tools import generate_tool_descriptions
from tools import (
    ClippingTool,
    DenoiseTool,
    AmplitudeNormalizeTool,
    LoudnessNormalizeTool,
    DCOffsetRemovalTool,
    SpectralNormalizeTool,
    TrimSilenceTool,
    PreEmphasisTool,
    SourceSeparationTool,
    ExtractTargetTool,
    RemoveTargetTool
)

tool_set =     [
        ClippingTool,
        DenoiseTool,
        # AmplitudeNormalizeTool,
        # LoudnessNormalizeTool,
        # DCOffsetRemovalTool,
        # SpectralNormalizeTool,
        # TrimSilenceTool,
        # PreEmphasisTool,
        # SourceSeparationTool,
        ExtractTargetTool,
        RemoveTargetTool
    ]

# tool_set = None

tools_description = generate_tool_descriptions(
    tool_set
)


In [14]:
subset = load_subset(subset_path)
random.shuffle(subset)
len(subset)

904

In [15]:
guidelines = (
    "- The downstream model is weak and may struggle with ambiguous or implicit information.Prefer tools that produce clear, explicit, structured outputs (e.g., transcripts, labeled events, timestamps) over raw or abstract representations."
    # "- When in doubt, select more tools rather than fewer — the downstream model cannot infer missing information on its own."
    "- Avoid assuming the downstream model can reason across modalities or fill in gaps from incomplete tool outputs."
    "- Consider multi-step reasoning: a tool may not directly answer the question, but its output combined with the parameters used gives the downstream model a clearer signal to work with."
    "- Audio manipulation tools can be valuable even if they do not produce text — their outputs reduce ambiguity and narrow the downstream model's search space."
    # "- Audio manipulation tools (e.g., source separation, denoising, clipping) can be valuable even if they do not produce text — their outputs reduce ambiguity and narrow the downstream model's search space."
"- When assigning importance_score, use the full range of 1-5:"
  "\t- 4-5: The question almost certainly cannot be answered without this tool."
  "\t- 3: The tool significantly narrows down the correct answer."
  "\t- 2: The tool provides useful context but the question might still be answerable without it."
  "\t- 1: The tool offers only marginal supplementary information."
)

In [16]:
from google.genai.types import Schema
tool_schema = Schema(
    type="array",
    items=Schema(
        type="object",
        properties={
            "tool":             Schema(type="string"),
            "role":             Schema(type="string", enum=["required", "helpful"]),
            "importance_score": Schema(type="integer"),
            "reason":           Schema(type="string"),
            "parameters":       Schema(type="object"),
        },
        required=["tool", "role", "importance_score", "reason"]
    )
)


In [17]:
gen_tool_prompt = '''
You are an expert system for planning tool usage in audio question answering.

Your goal is to select a set of tools that helps another weaker language model answer correctly.
The downstream model has limited reasoning ability and cannot process raw audio directly.
It relies entirely on the structured text outputs you provide — the more explicit and 
pre-processed the information, the better it can perform.

You will be given:
- A question about an audio clip
- Multiple-choice options
- An audio input
- A list of available tools with descriptions

---

### Your task:

1. Analyze what information is required to answer the question:
   - Speech content (ASR)
   - Sound events (e.g., dog barking, car honking)
   - Speaker identity or count
   - Temporal information (timestamps)
   - Acoustic properties (pitch, loudness, emotion)
   - Music-related information

2. Select tools that can help answer the question, either directly or indirectly:
   - Directly: tools that extract explicit information (e.g., transcripts, timestamps, labeled sound events)
   - Indirectly: tools that isolate, enhance, or transform the audio so that the downstream model can more easily identify the relevant information.  
   
For example:
   - If the question asks about vocal content (e.g., what was said, whether someone is speaking),
     consider using extract_source with label "vocals" to isolate the speech signal, making it
     easier for the downstream model to reason about speech-related properties.
   - If the question focuses on background sounds or music and vocals are interfering,
     consider using remove_source with label "vocals" to eliminate the speech layer, so the
     downstream model can reason more clearly about the remaining audio content.
   - If the question focuses on a specific time range, consider clipping the audio 
     to that segment so the downstream model can focus on the relevant portion 
     without being distracted by irrelevant content.
   - If the audio contains heavy background noise that may obscure speech or sound 
     events, consider denoising the audio first to improve the clarity of subsequent 
     tool outputs or downstream model reasoning.
   - If the question involves speech content (e.g., what was said, word count, 
     language spoken), ASR should be applied — and if the audio is noisy, consider 
     combining denoising before ASR to improve transcription accuracy.

3. You may select AT MOST {max_tools} tool(s) in total. Rank candidates by importance_score
   and keep only the highest-ranked ones. If fewer tools are sufficient, select fewer.

4. Avoid selecting tools that do not contribute useful information.

---

### Important Guidelines:

{guidelines}
---

### Context Passed to the Downstream Model:

The downstream model will receive the following information:
- The original question and multiple-choice options
- The tool calls you selected, including the tool name and parameters used
- The output returned by each tool

Therefore, you do not need to worry about whether the downstream model can trace 
where a piece of information comes from — it will have full visibility into both 
the tool parameters and their corresponding outputs.

Focus solely on selecting tools whose outputs provide information that is 
relevant and sufficient to answer the question.

---

### Question Information:

Question: {question}
Choices: {choices}
Audio: {audio_token}
Audio id: {audio_id}
Following are the descriptions of the tools you can use:
{tools_description}

### Output format:

[
    {{
        "tool": "ToolName",
        "role": "required" | "helpful",
        "importance_score": <integer from 1 to 10, where 10 means the question almost certainly cannot be answered correctly without this tool, and 1 means it provides only marginal supplementary information>,
        "reason": "Explain why this tool is useful and how its explicit output helps a weak downstream LLM — that cannot reason over raw audio — answer the question directly.",
        "parameters": {{"example_param": "value"}}
    }}
]
'''

output_note = '''Note: Always populate "parameters" with the specific configuration for the tool. Examples:
- Clipping:           {{"start_time": 2.0, "end_time": 6.0}}
- Extract source:     {{"label": "vocals"}}
- Remove source:      {{"label": "background music"}}
- ASR:                {{"language": "en"}}
- Denoising:          {{}}  (no parameters required)
'''

In [18]:
from tqdm import tqdm
from gemini.gemini import gemini_inference
from shutil import copyfile

# tools_description = generate_tool_descriptions()
thinking_level = "Low"

audio_token = "<AUDIO_TOKEN>"  # Placeholder for the actual audio token to be used in the prompt

audio_target_dir = '/home/u1501463/tool_use_LALM/dcase_audios'

max_tools = 2  # Maximum number of tools the LLM is allowed to select

tool_schedules = []

tool_cnt = {}

output_path = '../../tool_schedules.json'

for item in tqdm(subset):
    question = item.get("question", "")
    choices = item.get("choice", [])
    answer = item.get("answer", "")
    audio_path = item.get("audio_path", "")

    # Copy the audio file to the target directory
    copyfile(audio_path, f"{audio_target_dir}/{item.get('id', 'default')}.wav")

    gen_tool_oracle_prompt = gen_tool_prompt.format(
        guidelines=guidelines,
        question=question,
        choices=choices,
        audio_token=audio_token,
        tools_description=tools_description,
        audio_id=0,
        max_tools=max_tools,
    )

    # print(gen_tool_oracle_prompt)

    # break
    # print(question)
    # print(choices)
    # print(answer)
    # print(audio_path)
    output = gemini_inference(
        instruction=gen_tool_oracle_prompt,
        audio_token=audio_token,
        audio_path=audio_path,
        thinking_level=thinking_level,
        response_mime_type="application/json",
    )

    tool_schedule = json.loads(output.text)

    for tool in tool_schedule:
        tool_name = tool.get("tool", "UnknownTool")
        tool_cnt[tool_name] = tool_cnt.get(tool_name, 0) + 1

    print('\r', tool_cnt)
    tool_schedules.append((item, tool_schedule))
    with open(output_path, "w") as f:
        json.dump(tool_schedules, f, indent=4)


  0%|          | 0/904 [00:32<?, ?it/s]


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
tool_schedule